IQAE validating the T transition epsilon, noisless condition


In [4]:

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import json

from qiskit.circuit import QuantumCircuit
from qiskit import transpile
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, depolarizing_error
from qiskit_aer.primitives import Sampler as AerSampler
from qiskit_algorithms import IterativeAmplitudeEstimation, EstimationProblem
from qiskit.primitives import Sampler

sampler = Sampler()

print("All imports OK ✅")

p_true = 0.1  ########################################################################################################

class BernoulliA(QuantumCircuit):
    def __init__(self, probability):
        super().__init__(1)
        theta_p = 2 * np.arcsin(np.sqrt(probability))
        self.ry(theta_p, 0)

class BernoulliQ(QuantumCircuit):
    def __init__(self, probability):
        super().__init__(1)
        self._theta_p = 2 * np.arcsin(np.sqrt(probability))
        self.ry(2 * self._theta_p, 0)

    def power(self, k):
        q_k = QuantumCircuit(1)
        # ---------------------------------------------------------
        # REWRITTEN: Apply base rotation k times separated by barriers
        # to ensure the noise model acts k times.
        # ---------------------------------------------------------
        for _ in range(k):
            q_k.ry(2 * self._theta_p, 0)
            q_k.barrier() 
        return q_k

problem = EstimationProblem(
    state_preparation=BernoulliA(p_true),
    grover_operator=BernoulliQ(p_true),
    objective_qubits=[0],
)
print("Problem defined ✅")

All imports OK ✅
Problem defined ✅


C:\Users\Percy Cheng\AppData\Local\Temp\ipykernel_29692\204020335.py:14: DeprecationWarning: The class ``qiskit.primitives.sampler.Sampler`` is deprecated as of qiskit 1.2. It will be removed no earlier than 3 months after the release date. All implementations of the `BaseSamplerV1` interface have been deprecated in favor of their V2 counterparts. The V2 alternative for the `Sampler` class is `StatevectorSampler`.
  sampler = Sampler()


In [5]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 1 — Parameters + Helper Functions  (fixed)
# ══════════════════════════════════════════════════════════════════════════════

import numpy as np
from joblib import Parallel, delayed


R         = 1000
ALPHA     = 0.05
N_JOBS    = 18

TARGET         = 0.50
CONV_BAND = 0.02
LOCAL_HALF_WIDTH = 0.05   # warm-start lambda window: [p*_prev - 0.05, p*_prev + 0.05]
PRECISION      = 0.0001

epsilon_values = np.arange(0.0001, np.pi/8, 0.001).tolist()


def success_rate(k, n):
    return k / n


# ── Ceiling N_max ──────────────────────────────────────────────────────────────
# Rounds down to prevent from overshooting the theoretical limit
def N_max_Chernoff(epsilon, alpha):
    raw = (32 / (1 - 2 * np.sin(np.pi / 14)) ** 2) \
          * np.log((2 / alpha) * np.log2(np.pi / (4 * epsilon)))
    return int(np.floor(raw))

def N_max_CP(epsilon, alpha):
    raw = 69 * np.log((2 / alpha) * np.log2(np.pi / (4 * epsilon)))
    return int(np.floor(raw))

In [6]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 2 — Single run + Probe
# ══════════════════════════════════════════════════════════════════════════════

def single_run(p_noise, eps, N_SHOTS):
    """One IQAE run at a fixed shot budget. Returns 1 (success) or 0 (failure)."""
    try:
        iae     = IterativeAmplitudeEstimation(
            epsilon_target = eps,
            alpha          = ALPHA,
            sampler        = sampler,
            confint_method    = "beta" #chernoff" "beta"###################################################
        )
        result  = iae.estimate(problem)
        return 1 if abs(result.estimation_processed - p_true) <= eps else 0
    except Exception as e:
        print(f"[WARN] single_run failed: {type(e).__name__}: {e}")
        return 0


def probe(eps, p):
    """
    Run R independent IQAE runs at (eps, p) in parallel, at the ceiling
    shot budget N_max(eps, ALPHA).
    Returns (s_hat, lower_CI, upper_CI, half_width, N_SHOTS).
    """
    N_SHOTS = N_max_CP(eps, ALPHA)   ##########################################

    outcomes           = Parallel(n_jobs=N_JOBS)(
        delayed(single_run)(p, eps, N_SHOTS) for _ in range(R)
    )
    k                  = sum(outcomes)
    s_hat = success_rate(k, R)
    return s_hat, N_SHOTS

In [7]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 3 — Bisection
# ══════════════════════════════════════════════════════════════════════════════

def find_boundary(eps, p_low, p_high):
    """
    Bisect over p to find p* where success rate ≈ 50% ± CONV_BAND.
    Matches the pseudocode agreed in transcript exactly.
    """
    history = []

    while (p_high - p_low) > PRECISION:

        p_mid                      = (p_low + p_high) / 2.0
        s_hat, n_shots = probe(eps, p_mid)

        history.append({
            "p_try"       : p_mid,
            "s_hat"       : s_hat,
            "n_shots"     : n_shots,
        })

        print(f"    ε={eps:.4f} | p={p_mid:.4f} | "
              f"s={s_hat:.3f} | "
              f"N_shots={n_shots} | "
              f"window=[{p_low:.4f}, {p_high:.4f}]")

        if s_hat > TARGET + CONV_BAND:
            # ✅ Success rate too high → boundary is above p_mid → raise floor
            p_low  = p_mid

        elif s_hat < TARGET - CONV_BAND:
            # ✅ Success rate too low → boundary is below p_mid → lower ceiling
            p_high = p_mid

        else:
            # ✅ In band [0.45, 0.55] → converged
            print(f"    ✅ Converged at p={p_mid:.4f}")
            return {
                "epsilon"     : eps,
                "p_star"      : p_mid,
                "s_hat"       : s_hat,
                "n_shots"     : n_shots,
                "converged"   : True,
                "cliff"       : False,
                "history"     : history,
                "final_p_low" : p_low,
                "final_p_high": p_high,
                "uncertainty": (p_high - p_low) / 2.0,
            }

    p_best                     = (p_low + p_high) / 2.0
    s_hat, n_shots = probe(eps, p_best)

    print(f"    ⚠️  Precision limit at p={p_best:.4f} | s={s_hat:.3f}")
    return {
        "epsilon"     : eps,
        "p_star"      : p_best,
        "s_hat"       : s_hat,
        "n_shots"     : n_shots,
        "converged"   : False,
        "cliff"       : False,
        "history"     : history,
        "final_p_low" : p_low,
        "final_p_high": p_high,
        "uncertainty": (p_high - p_low) / 2.0,
    }

In [8]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 4 — Outer sweep (added: early termination if p* saturates at 1.0)
# ══════════════════════════════════════════════════════════════════════════════

def sweep_frontier(epsilon_list, local_half_width=LOCAL_HALF_WIDTH):

    results     = []
    p_star_prev = None

    for eps in epsilon_list:

        print(f"\n{'─'*60}")
        print(f"  ε = {eps:.4f}")
        print(f"{'─'*60}")

        # ── CASE 1: First epsilon ─────────────────────────────────────────────
        if p_star_prev is None:
            p_low  = 0.0
            p_high = 1.0
            cliff  = False
            print("  First ε — bisecting full range [0.0, 1.0]")

        # ── CASE 2: Subsequent epsilons ───────────────────────────────────────
        else:

            print(f"  Probing at previous p* = {p_star_prev:.4f} ...")
            s_probe, n_shots_probe = probe(eps, p_star_prev)
            print(f"  Probe: s={s_probe:.3f} "
                  f"| N_shots={n_shots_probe}")

            if abs(s_probe - TARGET) <= CONV_BAND:
                p_low  = max(0.0, p_star_prev - local_half_width)
                p_high = min(1.0, p_star_prev + local_half_width)
                cliff  = False
                print(f"  Near boundary — local refinement in "
                      f"[{p_low:.4f}, {p_high:.4f}]")

            elif s_probe > TARGET + CONV_BAND:
                p_low  = p_star_prev
                p_high = 1.0
                cliff  = True
                print(f"  ⚠️  Cliff detected (s={s_probe:.3f}) — "
                      f"searching [{p_low:.4f}, {p_high:.4f}]")

            else:
                p_low  = 0.0
                p_high = p_star_prev
                cliff  = False
                print(f"  ⚠️  Boundary dropped (s={s_probe:.3f}) — "
                      f"searching [{p_low:.4f}, {p_high:.4f}]")

        # ── Bisect ────────────────────────────────────────────────────────────
        result          = find_boundary(eps, p_low, p_high)
        result["cliff"] = cliff
        result["search_window_low"]  = p_low
        result["search_window_high"] = p_high
        result["used_local_refinement"] = (
            p_star_prev is not None
            and abs(result["search_window_low"]
                    - max(0.0, p_star_prev - local_half_width)) < 1e-12
            and abs(result["search_window_high"]
                    - min(1.0, p_star_prev + local_half_width)) < 1e-12
            if p_star_prev is not None else False
        )

        results.append(result)
        p_star_prev = result["p_star"]

        status = "✅ converged" if result["converged"] else "⚠️  precision limit"
        print(f"\n  {status} | p* = {result['p_star']:.4f} | "
              f"s = {result['s_hat']:.3f} | N_shots = {result['n_shots']}")


    return results

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 5 — Full run
# ══════════════════════════════════════════════════════════════════════════════

frontier_results = sweep_frontier(epsilon_values)

print(f"\n{'═'*60}")
print(f"  ✅ Frontier sweep complete — {len(frontier_results)} points")
print(f"{'═'*60}")

for r in frontier_results:
    cliff_tag = " ⚠️  CLIFF" if r["cliff"] else ""
    conv_tag  = "✅" if r["converged"] else "⚠️ "
    print(f"  {conv_tag} ε={r['epsilon']:.5f} | "
          f"p*={r['p_star']:.4f} ± {r['uncertainty']:.4f} | "
          f"s={r['s_hat']:.3f} | "
          f"N={r['n_shots']}{cliff_tag}")


────────────────────────────────────────────────────────────
  ε = 0.0001
────────────────────────────────────────────────────────────
  First ε — bisecting full range [0.0, 1.0]
    ε=0.0001 | p=0.5000 | s=1.000 | N_shots=431 | window=[0.0000, 1.0000]
    ε=0.0001 | p=0.7500 | s=1.000 | N_shots=431 | window=[0.5000, 1.0000]
    ε=0.0001 | p=0.8750 | s=1.000 | N_shots=431 | window=[0.7500, 1.0000]
    ε=0.0001 | p=0.9375 | s=1.000 | N_shots=431 | window=[0.8750, 1.0000]
    ε=0.0001 | p=0.9688 | s=1.000 | N_shots=431 | window=[0.9375, 1.0000]
    ε=0.0001 | p=0.9844 | s=1.000 | N_shots=431 | window=[0.9688, 1.0000]
    ε=0.0001 | p=0.9922 | s=1.000 | N_shots=431 | window=[0.9844, 1.0000]
    ε=0.0001 | p=0.9961 | s=1.000 | N_shots=431 | window=[0.9922, 1.0000]
    ε=0.0001 | p=0.9980 | s=1.000 | N_shots=431 | window=[0.9961, 1.0000]
    ε=0.0001 | p=0.9990 | s=1.000 | N_shots=431 | window=[0.9980, 1.0000]
    ε=0.0001 | p=0.9995 | s=1.000 | N_shots=431 | window=[0.9990, 1.0000]
    ε=

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 6+7 — Plot + Export (uncertainty taken directly from result dict)
# ══════════════════════════════════════════════════════════════════════════════

import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np
from datetime import datetime
import os
import math

# ── Build plotting arrays ─────────────────────────────────────────────────────
eps_vals     = [r["epsilon"]   for r in frontier_results]
p_star_vals  = [r["p_star"]    for r in frontier_results]
conv_mask    = [r["converged"] for r in frontier_results]
cliff_mask   = [r.get("cliff", False) for r in frontier_results]
n_shots_vals = [r["n_shots"]   for r in frontier_results if r.get("n_shots") is not None]

# ── Pull uncertainty directly from stored result ──────────────────────────────
# find_boundary() stores: "uncertainty": (p_high - p_low) / 2.0
# Symmetric bar: lower_err = upper_err = uncertainty
uncertainties = [r["uncertainty"] for r in frontier_results]
yerr = np.array([uncertainties, uncertainties])   # shape (2, N)

# ── Figure setup ──────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8.5, 5.2))

# ── Error bars on p* ──────────────────────────────────────────────────────────
ax.errorbar(
    eps_vals,
    p_star_vals,
    yerr=yerr,
    fmt='none',
    ecolor='#2166ac',
    alpha=0.30,
    capsize=2.5,
    elinewidth=0.9,
    label='_nolegend_'
)

# ── Converged points ──────────────────────────────────────────────────────────
conv_eps = [e for e, cv in zip(eps_vals, conv_mask) if cv]
conv_p   = [p for p, cv in zip(p_star_vals, conv_mask) if cv]
ax.scatter(
    conv_eps, conv_p,
    color='#2166ac', s=34, zorder=3,
    label=r'Boundary $p^{*}(\epsilon)$'
)

# ── Not-converged points ──────────────────────────────────────────────────────
nc_eps = [e for e, cv in zip(eps_vals, conv_mask) if not cv]
nc_p   = [p for p, cv in zip(p_star_vals, conv_mask) if not cv]
if nc_eps:
    ax.scatter(
        nc_eps, nc_p,
        color='#d73027', marker='x', s=48,
        linewidths=1.2, zorder=4,
        label='Precision limit'
    )

# ── Cliff-flagged points ──────────────────────────────────────────────────────
cliff_eps = [e for e, cf in zip(eps_vals, cliff_mask) if cf]
cliff_p   = [p for p, cf in zip(p_star_vals, cliff_mask) if cf]
if cliff_eps:
    ax.scatter(
        cliff_eps, cliff_p,
        facecolors='none', edgecolors='#542788',
        s=70, linewidths=1.1, zorder=5,
        label='Cliff flagged'
    )

# ── Axes & styling ────────────────────────────────────────────────────────────
ax.set_xscale('linear')
ax.xaxis.set_major_formatter(ticker.FormatStrFormatter('%.3f'))
ax.tick_params(axis='x', labelsize=8)

ax.set_xlabel(r'Target precision $\epsilon$', fontsize=11)
ax.set_ylabel(r'Critical noise level $p^{*}$', fontsize=11)

if n_shots_vals:
    n_shots_label = (rf'$N_{{\mathrm{{shot}}}} \in '
                     rf'[{min(n_shots_vals)}, {max(n_shots_vals)}]$ (ceiling)')
else:
    n_shots_label = rf'$N_{{\mathrm{{shot}}}}$ = <UNSET>'

ax.set_title(
    r'IQAE noise-threshold frontier' + '\n' +
    rf'Adaptive bisection, {n_shots_label}, $R={R}$, $\alpha={ALPHA}$',
    fontsize=11, pad=10
)

ax.legend(fontsize=8.2, framealpha=0.92, edgecolor='0.75',
          loc='upper left', ncol=1)
ax.grid(True, which='both', alpha=0.22, linewidth=0.5)
ax.set_xlim(min(eps_vals), max(eps_vals))
ax.set_ylim(0.0, 1.0)

for spine in ['top', 'right']:
    ax.spines[spine].set_visible(False)

plt.tight_layout()
plt.show()


# ══════════════════════════════════════════════════════════════════════════════
# Export
# ══════════════════════════════════════════════════════════════════════════════

def _safe_get(name, default="<UNSET>"):
    return globals().get(name, default)

def _format_scalar(x):
    try:
        if isinstance(x, (np.floating, np.integer)):
            return x.item()
    except Exception:
        pass
    return x

def _format_epsilon_values_as_linspace(vals):
    """Return a compact np.linspace(...) string if the grid is evenly spaced."""
    try:
        vals = list(vals)
        if len(vals) >= 2:
            start = float(vals[0])
            stop  = float(vals[-1])
            num   = len(vals)
            step  = (stop - start) / (num - 1)
            evenly_spaced = all(
                math.isclose(float(vals[i+1]) - float(vals[i]),
                             step, rel_tol=1e-9, abs_tol=1e-12)
                for i in range(num - 1)
            )
            if evenly_spaced:
                return f"np.linspace({start:.6f}, {stop:.6f}, {num}).tolist()"
    except Exception:
        pass
    return repr(vals)

def export_frontier_points_txt(
    frontier_results,
    filename="iqae_frontier_points_export.txt",
    extra_notes=None,
):
    epsilon_values = _safe_get("epsilon_values")

    params = {
        "timestamp"       : datetime.now().isoformat(timespec="seconds"),
        "p_true"          : _format_scalar(_safe_get("p_true")),
        "R"               : _format_scalar(_safe_get("R")),
        "ALPHA"           : _format_scalar(_safe_get("ALPHA")),
        "N_JOBS"          : _format_scalar(_safe_get("N_JOBS")),
        "TARGET"          : _format_scalar(_safe_get("TARGET")),
        "CONV_BAND"       : _format_scalar(_safe_get("CONV_BAND")),
        "PRECISION"       : _format_scalar(_safe_get("PRECISION")),
        "epsilon_values"  : (_format_epsilon_values_as_linspace(epsilon_values)
                             if epsilon_values != "<UNSET>" else "<UNSET>"),
        "SINGLE_QUBIT_GATES": _format_scalar(_safe_get("SINGLE_QUBIT_GATES")),
        "N_shots_mode"    : "ceiling (N_max(eps, ALPHA), varies per epsilon — see n_shots column)",
    }

    for name in ["r_min", "Z_95", "backend_name", "noise_model_name", "sampler_type"]:
        if name in globals():
            params[name] = _format_scalar(globals()[name])

    lines = []
    lines.append("=" * 80)
    lines.append("IQAE FRONTIER DATA POINTS EXPORT")
    lines.append("=" * 80)
    lines.append("")
    lines.append("[RUN PARAMETERS]")
    for k, v in params.items():
        lines.append(f"{k}: {v}")
    if extra_notes is not None:
        lines.append(f"extra_notes: {extra_notes}")
    lines.append("")

    # ── Data table now includes uncertainty column ─────────────────────────
    lines.append("[FRONTIER DATA POINTS]")
    header = "epsilon | p_star | uncertainty | s_hat | n_shots | converged | cliff"
    lines.append(header)
    lines.append("-" * len(header))

    for r in frontier_results:
        eps         = float(r.get("epsilon",     float("nan")))
        p_star      = float(r.get("p_star",      float("nan")))
        uncertainty = float(r.get("uncertainty", float("nan")))
        s_hat       = float(r.get("s_hat",       float("nan")))
        n_shots     = r.get("n_shots",    "<UNSET>")
        converged   = r.get("converged",  False)
        cliff       = r.get("cliff",      False)

        lines.append(
            f"{eps:.6f} | {p_star:.6f} | {uncertainty:.6f} | "
            f"{s_hat:.6f} | {n_shots} | {converged} | {cliff}"
        )

    lines.append("")

    with open(filename, "w", encoding="utf-8") as f:
        f.write("\n".join(lines))

    print(f"✅ Exported to: {os.path.abspath(filename)}")
    return os.path.abspath(filename)


# ── Run export ────────────────────────────────────────────────────────────────
export_path = export_frontier_points_txt(
    frontier_results,
    filename="adaptive_searchCP, a = 0.1, noiseless.txt",
    extra_notes="Compact export: parameters at top, frontier data points only."
)
print(export_path)

In [ ]:
import re
import numpy as np

def parse_iqae_ceiling_file(filepath):
    """
    Parse an IQAE ceiling-mode frontier data export file.
    Extracts the first 5 columns: epsilon, p_star, uncertainty, s_hat, n_shots

    Returns:
        epsilon      : np.ndarray
        p_star       : np.ndarray
        uncertainty  : np.ndarray
        s_hat        : np.ndarray
        n_shots      : np.ndarray (int)
    """
    epsilons      = []
    p_stars       = []
    uncertainties = []
    s_hats        = []
    n_shots       = []

    with open(filepath, "r", encoding="utf-8", errors="replace") as f:  # ← fix here
        lines = f.readlines()

    in_data_section = False

    for line in lines:
        line = line.strip()

        # ── Detect start of data section ──────────────────────────────────────
        if line.startswith("epsilon | p_star"):
            in_data_section = True
            continue  # skip the header row

        # ── Skip separator line ───────────────────────────────────────────────
        if in_data_section and set(line) <= set("-"):
            continue

        # ── Parse data rows ───────────────────────────────────────────────────
        if in_data_section and "|" in line:
            parts = [p.strip() for p in line.split("|")]
            epsilons.append(float(parts[0]))
            p_stars.append(float(parts[1]))
            uncertainties.append(float(parts[2]))
            s_hats.append(float(parts[3]))
            n_shots.append(int(parts[4]))

    return (
        np.array(epsilons),
        np.array(p_stars),
        np.array(uncertainties),
        np.array(s_hats),
        np.array(n_shots),
    )


# ── Usage ─────────────────────────────────────────────────────────────────────
ALPHA = 0.05
epsilon, p_star, p_uncertainty, s_hat, n_shots = parse_iqae_ceiling_file(
    "adaptive_searchCP, a = 0.1, noiseless.txt"
)

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL A
# ══════════════════════════════════════════════════════════════════════════════
def _safe_float(x):
    return float(x) if x is not None else None

def _safe_list(x):
    return list(x) if x is not None else None

def _safe_list_of_lists(x):
    return [list(item) for item in x] if x is not None else None

def _safe_delta_theta_list(theta_intervals):
    """
    Compute Δθ_t = θ_hi - θ_lo for every round t = 1 … T.
    Returns a list of floats (one per round), or None if unavailable.
    Malformed individual rounds are stored as None to preserve indexing.
    """
    if theta_intervals is None or len(theta_intervals) == 0:
        return None
    result = []
    for interval in theta_intervals:
        if interval is not None and len(interval) == 2:
            result.append(float(interval[1] - interval[0]))
        else:
            result.append(None)   # malformed round — keep slot
    return result


def single_diag_run_full(eps, p_noise, n_shots):
    """
    One IQAE run at (eps, p_noise) with shot budget n_shots.
    Captures the full IterativeAmplitudeEstimationResult, excluding
    circuit_results (raw statevector/counts — not needed here).
    Each field is converted defensively so a single None/missing field
    doesn't discard the whole run.
    """
    try:
        iae     = IterativeAmplitudeEstimation(
            epsilon_target = eps,
            alpha          = ALPHA,
            sampler        = sampler,
            confint_method = "beta",  # chernoff or beta (CH or CP)
        )
        result = iae.estimate(problem)

        powers   = list(result.powers) if result.powers is not None else []
        K_values = [2 * k + 1 for k in powers]

        estimate = _safe_float(result.estimation_processed)
        success  = int(estimate is not None and abs(estimate - p_true) <= eps)

        # ── theta intervals — build once, derive delta_theta from it ──────────
        theta_intervals    = _safe_list_of_lists(result.theta_intervals)
        delta_theta_list   = _safe_delta_theta_list(theta_intervals)

        return {
            "success"                      : success,
            "powers"                       : powers,
            "K_values"                     : K_values,
            "T_eps"                        : len(powers),
            "max_K"                        : max(K_values) if K_values else None,
            "sum_K"                        : sum(K_values) if K_values else None,

            # amplitude-domain results
            "estimation"                   : estimate,
            "signed_error"                 : (estimate - p_true) if estimate is not None else None,
            "confidence_interval"          : _safe_list(result.confidence_interval),
            "confidence_interval_processed": _safe_list(result.confidence_interval_processed),
            "estimate_intervals"           : _safe_list_of_lists(result.estimate_intervals),

            # angle-domain results
            "theta_intervals"              : theta_intervals,
            "delta_theta_list"             : delta_theta_list,              # NEW: [Δθ_1, …, Δθ_T]

            # schedule / information mechanics
            "ratios"                       : _safe_list(result.ratios),
            "num_oracle_queries"           : int(result.num_oracle_queries) if result.num_oracle_queries is not None else None,

            # precision bookkeeping
            "epsilon_target"               : _safe_float(result.epsilon_target),
            "epsilon_estimated"            : _safe_float(result.epsilon_estimated),
            "epsilon_estimated_processed"  : _safe_float(result.epsilon_estimated_processed),
            "alpha_echo"                   : _safe_float(result.alpha),

            # sampling bookkeeping
            "shots"                        : result.shots,
            "n_shots_requested"            : n_shots,

            "error_message"                : None,
        }

    except Exception as e:
        err_str = f"{type(e).__name__}: {e}"
        print(f"[WARN] single_diag_run_full failed: {err_str}")
        return {
            "success": 0, "powers": [], "K_values": [], "T_eps": 0,
            "max_K": None, "sum_K": None,
            "estimation": None, "signed_error": None,
            "confidence_interval": None, "confidence_interval_processed": None,
            "estimate_intervals": None,
            "theta_intervals"              : None,
            "delta_theta_list"             : None,                          # NEW
            "ratios": None, "num_oracle_queries": None,
            "epsilon_target": eps, "epsilon_estimated": None,
            "epsilon_estimated_processed": None, "alpha_echo": ALPHA,
            "shots": None, "n_shots_requested": n_shots,
            "error_message": err_str,
        }

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL B  - Simulation sweep: run R runs per (ε, p*) point, collect full results
# ══════════════════════════════════════════════════════════════════════════════

from collections import Counter
from joblib import Parallel, delayed
import numpy as np

R = 1000   # runs per point

diag_results = []

for eps, p_star_val in zip(epsilon, p_star):
    n_shots_diag = int(N_max_CP(eps, ALPHA))   # ← Cp or CH

    print(f"\n{'─'*60}")
    print(f"ε={eps:.6f} | p*={p_star_val:.6f} | N_shots={n_shots_diag} "
          f"| running {R} runs ...")

    runs = Parallel(n_jobs=N_JOBS)(
        delayed(single_diag_run_full)(eps, p_star_val, n_shots_diag)
        for _ in range(R)
    )

    # ── Live per-run schedule output ──────────────────────────────────────────
    for i, run in enumerate(runs):
        if run["error_message"] is not None:
            print(f"  run {i+1:>3} | ERROR: {run['error_message']}")
            continue

        status  = "✅" if run["success"] else "❌"
        err_str = f"{run['signed_error']:.6f}" if run["signed_error"] is not None else "N/A"

        # final Δθ — last element of the stored list
        dtl     = run.get("delta_theta_list")
        dth_str = f"{dtl[-1]:.6f}" if (dtl and dtl[-1] is not None) else "N/A"

        print(f"  run {i+1:>3} | {status} | T={run['T_eps']:>2} | "
              f"K_values={run['K_values']} | signed_error={err_str} | "
              f"Δθ_final={dth_str} | Δθ_seq={dtl}")

    # ── Aggregate ─────────────────────────────────────────────────────────────
    valid_runs   = [r for r in runs if r["T_eps"] > 0]
    success_rate = np.mean([r["success"] for r in runs])

    T_values    = [r["T_eps"] for r in valid_runs]
    maxK_values = [r["max_K"] for r in valid_runs]
    sumK_values = [r["sum_K"] for r in valid_runs]
    err_values  = [r["signed_error"] for r in valid_runs
                   if r["signed_error"] is not None]

    # ── delta_theta aggregation — from delta_theta_list ───────────────────────
    # final Δθ per run (last element of each run's list)
    dtheta_final_values = [
        r["delta_theta_list"][-1]
        for r in valid_runs
        if r.get("delta_theta_list") and r["delta_theta_list"][-1] is not None
    ]

    # full Δθ sequences — one list per run (for downstream convergence plots)
    dtheta_sequences = [
        r["delta_theta_list"]
        for r in valid_runs
        if r.get("delta_theta_list") is not None
    ]

    schedule_counts = Counter(tuple(r["K_values"]) for r in valid_runs)
    most_common_schedule, most_common_count = (
        schedule_counts.most_common(1)[0] if schedule_counts else ((), 0)
    )

    diag_results.append({
        "epsilon"              : eps,
        "p_star"               : p_star_val,
        "n_shots"              : n_shots_diag,
        "success_rate"         : success_rate,
        "T_mean"               : float(np.mean(T_values))    if T_values    else None,
        "T_std"                : float(np.std(T_values))     if T_values    else None,
        "max_K_mean"           : float(np.mean(maxK_values)) if maxK_values else None,
        "sum_K_mean"           : float(np.mean(sumK_values)) if sumK_values else None,
        "mean_signed_error"    : float(np.mean(err_values))  if err_values  else None,
        # ── Δθ final (scalar summary) ─────────────────────────────────────────
        "delta_theta_mean"     : float(np.mean(dtheta_final_values))   if dtheta_final_values else None,
        "delta_theta_std"      : float(np.std(dtheta_final_values))    if dtheta_final_values else None,
        "delta_theta_median"   : float(np.median(dtheta_final_values)) if dtheta_final_values else None,
        "n_dtheta_collected"   : len(dtheta_final_values),
        # ── Δθ sequences (full per-round lists) ──────────────────────────────
        "delta_theta_sequences": dtheta_sequences,   # list of lists — one per valid run
        # ─────────────────────────────────────────────────────────────────────
        "most_common_schedule" : list(most_common_schedule),
        "most_common_count"    : most_common_count,
        "n_valid_runs"         : len(valid_runs),
        "raw_runs"             : runs,
    })

    print(f"  → success_rate={success_rate:.3f} | "
          f"Δθ_final mean={diag_results[-1]['delta_theta_mean']:.6f} ± "
          f"{diag_results[-1]['delta_theta_std']:.6f} | "
          f"most_common_schedule={list(most_common_schedule)} "
          f"({most_common_count}/{len(valid_runs)})")

print("\n✅ Full diagnostic sweep complete.")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL C  (updated — uses delta_theta_list from Cell A)
# ══════════════════════════════════════════════════════════════════════════════

import numpy as np

eps_out              = []
p_star_out           = []
T_out                = []
K_values_out         = []
ratios_out           = []
success_out          = []
n_shots_diag_out     = []
signed_error_out     = []   # per-run signed error        (None if unavailable)
error_message_out    = []   # per-run error message       (None if clean run)
delta_theta_out      = []   # final Δθ per run            (None if unavailable)
delta_theta_list_out = []   # full Δθ sequence per run    (None if unavailable)

# ── Flatten all runs across all epsilon points ────────────────────────────────
for point in diag_results:
    eps_val          = point["epsilon"]
    p_star_val       = point["p_star"]
    n_shots_diag_val = point["n_shots"]

    for run in point["raw_runs"]:
        eps_out.append(eps_val)
        p_star_out.append(p_star_val)
        T_out.append(run["T_eps"])
        K_values_out.append(run["K_values"])
        ratios_out.append(run["ratios"])
        success_out.append(bool(run["success"]))
        n_shots_diag_out.append(n_shots_diag_val)
        signed_error_out.append(run.get("signed_error"))       # None if errored
        error_message_out.append(run.get("error_message"))     # None if clean

        # ── Δθ — read directly from Cell A's stored list ─────────────────────
        dtl = run.get("delta_theta_list")
        delta_theta_list_out.append(dtl)                       # full sequence or None
        delta_theta_out.append(
            dtl[-1] if (dtl and dtl[-1] is not None) else None # final value  or None
        )

# ── Convert to numpy arrays ───────────────────────────────────────────────────
eps_arr              = np.array(eps_out)
p_star_arr           = np.array(p_star_out)
T_arr                = np.array(T_out)
K_values_arr         = np.array(K_values_out,          dtype=object)  # ragged
ratios_arr           = np.array(ratios_out,             dtype=object)  # ragged
success_arr          = np.array(success_out,            dtype=bool)
n_shots_diag_arr     = np.array(n_shots_diag_out,       dtype=int)
signed_error_arr     = np.array(signed_error_out,       dtype=object)  # float or None
error_message_arr    = np.array(error_message_out,      dtype=object)  # str   or None
delta_theta_arr      = np.array(delta_theta_out,        dtype=object)  # float or None
delta_theta_list_arr = np.array(delta_theta_list_out,   dtype=object)  # list  or None

# ── Summary printout ──────────────────────────────────────────────────────────
n_total      = len(eps_arr)
n_errored    = int(np.sum([m is not None for m in error_message_out]))
n_success    = int(np.sum(success_arr))
n_fail_clean = int(np.sum(~success_arr)) - n_errored
n_dtheta     = int(np.sum([v is not None for v in delta_theta_out]))
n_dtheta_seq = int(np.sum([v is not None for v in delta_theta_list_out]))

print(f"Exported {n_total} run records across {len(diag_results)} epsilon points.")
print(f"  Failed runs (error_message set)  : {n_errored}")
print(f"  Successful runs                  : {n_success}")
print(f"  Failed (no error, just ❌)       : {n_fail_clean}")
print(f"  Runs with valid Δθ final         : {n_dtheta}     / {n_total}")
print(f"  Runs with valid Δθ sequence      : {n_dtheta_seq} / {n_total}")

# ── Save to disk ──────────────────────────────────────────────────────────────
np.savez(
    "ceilingsweepCP_Nmax, a = 0.1, noiseless.npz",
    epsilon          = eps_arr,
    p_star           = p_star_arr,
    T                = T_arr,
    K_values         = K_values_arr,
    ratios           = ratios_arr,
    success          = success_arr,
    n_shots_diag     = n_shots_diag_arr,
    signed_error     = signed_error_arr,
    error_message    = error_message_arr,
    delta_theta      = delta_theta_arr,        # scalar final Δθ per run
    delta_theta_list = delta_theta_list_arr,   # full [Δθ_1, …, Δθ_T] per run
    allow_pickle     = True,
)

print("✅ Saved — all per-run fields recorded.")